# Microsoft Foundry Worksheet
**NAVTTC / NIAI AI/ML Trainer Cohort**. Runs in Colab or Jupyter, plain Python. **Tasks 1 to 5 need no Azure account and no API key.** Fill every `None` / `# YOUR CODE` area, then run top to bottom. Check cells print `OK` or `FIX`.

You practise the parts around a hosted model: cost, messages, prompts, privacy and checking answers. Part B is the portal exercise (needs Azure access; steps follow Microsoft's docs and were not run by the author).

In [ ]:
import re, os
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))
print("ready")

## Task 1: What will a chat cost?

You are billed for tokens. Write `chat_cost(input_tokens, output_tokens, in_price, out_price)` where prices are **per 1 million tokens**. Prices below are **placeholders**: replace them with the real prices from the current pricing page before using the numbers for anything.

In [ ]:
IN_PRICE = 0.50     # placeholder: price per 1M input tokens
OUT_PRICE = 2.00    # placeholder: price per 1M output tokens
def chat_cost(input_tokens, output_tokens, in_price, out_price):
    return None      # YOUR CODE

In [ ]:
one = chat_cost(500, 200, IN_PRICE, OUT_PRICE)
month = one * 10_000
print(f"one chat: ${one:.6f}   10,000 chats: ${month:.2f}   (placeholder prices)")
check("one chat", abs(one - 0.00065) < 1e-12, "500/1e6*0.5 + 200/1e6*2.0")
check("10,000 chats", abs(month - 6.5) < 1e-9, "")
check("longer answers cost more", chat_cost(500, 2000, IN_PRICE, OUT_PRICE) > one, "")

## Task 2: Build the messages

A chat call sends a list of messages. Each is a dict with `role` (`system` or `user`) and `content`. Write `build_messages(system_text, user_text)` returning the two-message list, system first.

In [ ]:
def build_messages(system_text, user_text):
    return None      # YOUR CODE

In [ ]:
m = build_messages("You are a careful fraud analyst.", "Why is a 5,000 payment risky?")
check("two messages", isinstance(m, list) and len(m) == 2, "")
check("system first", m[0]["role"] == "system" and m[1]["role"] == "user", "")
check("content kept", m[1]["content"].startswith("Why is"), "")

## Task 3: A prompt template with four parts

Write `make_prompt(amount, usual, country_is_new)` returning one string with: a **role** line, a **task** line, the **context** (the numbers), and a **format/limits** line that says to say 'I do not know' if unsure.

In [ ]:
def make_prompt(amount, usual, country_is_new):
    return ""        # YOUR CODE: use an f-string

In [ ]:
p = make_prompt(5000, 100, True)
print(p)
check("has role", "Role" in p, "")
check("has numbers", "5000" in p and "100" in p, "put the context in the prompt")
check("allows I do not know", "I do not know" in p, "give the model a way out")

## Task 4: Remove private data before sending

Never send card numbers or phone numbers to a model without a reason. Write `clean(text)` that replaces any run of 13 to 16 digits (spaces or dashes allowed between digits) with `[CARD]`, and emails with `[EMAIL]`. Use `re.sub`.

In [ ]:
def clean(text):
    return text      # YOUR CODE

In [ ]:
t = "Customer ali@example.com paid with 4111 1111 1111 1111 for 5000."
c = clean(t)
print(c)
check("card removed", "4111" not in c and "[CARD]" in c, "")
check("email removed", "ali@example.com" not in c and "[EMAIL]" in c, "")
check("amount kept", "5000" in c, "do not remove the amount")
print("This is a simple filter. Real systems need stronger tools and a privacy review.")

## Task 5: Check an answer, do not trust it

A fake model gave five answers about the payment (amount 5,000, usual 100, new country). Write `check_answer(text)` that returns a score from 0 to 3: +1 if it mentions the amount (`5000` or `5,000`), +1 if it mentions the new country, +1 if it does **not** claim something we never gave it (it must not contain the word `stolen`). Then find which answer is best.

In [ ]:
answers = [
    "The payment of 5,000 is far above the usual 100 and comes from a new country.",
    "This card was stolen, so the payment is fraud.",
    "The amount is high.",
    "Payment 5000 from a new country is unusual compared with 100 normally.",
    "I do not know.",
]
def check_answer(text):
    return 0         # YOUR CODE
scores = [check_answer(a) for a in answers]

In [ ]:
print(scores)
check("first answer scores 3", scores[0] == 3, "amount + new country + no 'stolen'")
check("invented claim loses a point", scores[1] == 0 or scores[1] < 3, "'stolen' was never given")
best = scores.index(max(scores))
check("a best answer is found", scores[best] == 3, "")
print("Answer 5 ('I do not know') scored", scores[4], "- honest but unhelpful. A keyword score is a blunt tool: discuss what it misses.")

## Part B: the portal exercise (needs Azure access)
These steps follow Microsoft's public documentation at the time of writing. **Menu names change; they were not run by the author.** Do a dry run first.

1. Sign in at **ai.azure.com**. Create a project (or join your trainer's shared project). Under *Advanced options* you can set the resource group and region. If you see a policy error such as `RequestDisallowedByAzure`, your subscription limits regions: ask the trainer.
2. **Discover, then Models:** search for a small model and **Deploy** with default settings. Write down the **deployment name**.
3. In the **playground**, ask the same question with three different system messages (formal, very short, step-by-step). Write one line on how each answer differed.
4. **Build, then Agents, then New agent:** pick your deployment, write instructions for a fraud-analyst helper, save, and ask three questions. Make one question use data you did **not** give it. Did it say it did not know?
5. Record: tokens used, anything wrong, and what you would test next.
6. **Clean up:** delete deployments you no longer need to avoid charges.

**Reflection (2 sentences each):** Which prompt change helped most? What data would you never send to a hosted model?

## Optional code (not executed here)
Copy the current sample code from your project page in the portal. The general pattern is: read the endpoint and key from **environment variables**, create a client, send `build_messages(...)`, print the reply. Never paste your key into a notebook cell or a GitHub repository.